In [2]:
from tqdm.auto import tqdm
import pandas as pd
from typing import Optional, List, Tuple
import json
import datasets

/Users/abiodunsotunde/Documents/Personal/learning/AI-Engineering/Python/chapter-3/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ds = load_dataset("davidstap/biblenlp-corpus-mmteb", 'eng-deu')
ds = datasets.load_dataset('json',data_files='./data/en_kjv.json', split='train')
ds

Dataset({
    features: ['abbrev', 'name', 'chapters'],
    num_rows: 66
})

In [4]:
len(ds[0]['chapters'][0])

31

In [5]:
df = ds.to_pandas()
df

,abbrev,name,chapters
0,gn,Gênesis,[[In the beginning God created the heaven and ...
1,ex,Êxodo,[[Now these are the names of the children of I...
2,lv,Levítico,"[[And the LORD called unto Moses, and spake un..."
3,nm,Números,[[And the LORD spake unto Moses in the wildern...
4,dt,Deuteronômio,[[These be the words which Moses spake unto al...
...,...,...,...
61,1jo,1 João,"[[That which was from the beginning, which we ..."
62,2jo,2 João,[[The elder unto the elect lady and her childr...
63,3jo,3 João,"[[The elder unto the wellbeloved Gaius, whom I..."
64,jd,Judas,"[[Jude, the servant of Jesus Christ, and broth..."


In [21]:
# [doc for doc in ds]

In [6]:
from langchain_core.documents import Document as LangchainDocument
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = []

for book in ds:
    chapters = []
    chapter_index = 0
    joined_chapter = ''
    for chapter in book['chapters']:
        joined_verse = '\n'.join(chapter)
        # chapters.append(joined_verse)
        chapter_index += 1

    # for _chapter in chapters:
        joined_chapter += f'--CHAPTER {chapter_index}--\n' + joined_verse

    docs.append(
        LangchainDocument(page_content=joined_chapter, metadata={"chapter": book["name"]})
    )

docs[0]


# langchain_docs = [LangchainDocument(page_content='\'.join() '--CHAPTER--'.join() for chapter in book , metadata={"chapter": book["name"]}) for book in ds]
# langchain_docs

Document(metadata={'chapter': 'Gênesis'}, page_content='--CHAPTER 1--\nIn the beginning God created the heaven and the earth.\nAnd the earth was without form, and void; and darkness was upon the face of the deep. And the Spirit of God moved upon the face of the waters.\nAnd God said, Let there be light: and there was light.\nAnd God saw the light, that it was good: and God divided the light from the darkness.\nAnd God called the light Day, and the darkness he called Night. And the evening and the morning were the first day.\nAnd God said, Let there be a firmament in the midst of the waters, and let it divide the waters from the waters.\nAnd God made the firmament, and divided the waters which were under the firmament from the waters which were above the firmament: and it was so.\nAnd God called the firmament Heaven. And the evening and the morning were the second day.\nAnd God said, Let the waters under the heaven be gathered together unto one place, and let the dry land appear: and it

In [19]:
from langchain_core.documents import Document as LangchainDocument
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs_as_mark = ''

for book in ds:
    print(book['name'])
    if book['name'] == 'Êxodo':
        break
    chapter_index = 0
    joined_chapter = ''
    for chapter in book['chapters']:
        joined_verse = ''
        for index, verse in enumerate(chapter):
            joined_verse += f'- ' + verse + '\n'
            
        # chapters.append(joined_verse)
        chapter_index += 1

    # for _chapter in chapters:
        joined_chapter += f'## Chapter {chapter_index}\n\n' + joined_verse + '\n\n'

    docs_as_mark += f'# {book["name"]}\n\n' + joined_chapter + '\n\n'

print(docs_as_mark)

Gênesis
Êxodo
# Gênesis

## Chapter 1

- In the beginning God created the heaven and the earth.
- And the earth was without form, and void; and darkness was upon the face of the deep. And the Spirit of God moved upon the face of the waters.
- And God said, Let there be light: and there was light.
- And God saw the light, that it was good: and God divided the light from the darkness.
- And God called the light Day, and the darkness he called Night. And the evening and the morning were the first day.
- And God said, Let there be a firmament in the midst of the waters, and let it divide the waters from the waters.
- And God made the firmament, and divided the waters which were under the firmament from the waters which were above the firmament: and it was so.
- And God called the firmament Heaven. And the evening and the morning were the second day.
- And God said, Let the waters under the heaven be gathered together unto one place, and let the dry land appear: and it was so.
- And God cal

In [ ]:
total_chars = sum(len(doc.page_content) for doc in docs)
print(f"Total characters: {total_chars}")
# print(docs[0].page_content[:500])

Total characters: 4160598
--CHAPTER 1--
In the beginning God created the heaven and the earth.
And the earth was without form, and void; and darkness was upon the face of the deep. And the Spirit of God moved upon the face of the waters.
And God said, Let there be light: and there was light.
And God saw the light, that it was good: and God divided the light from the darkness.
And God called the light Day, and the darkness he called Night. And the evening and the morning were the first day.
And God said, Let there be a fi


In [ ]:
# chunk document based spliter using chapter then using text based spliter using resursive split

# Foo

## Bar

Hi this is Jim  
Hi this is Joe

 ## Baz
 
Hi this is Molly'

In [ ]:
# MD splits
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Book"),
    ("##", "Chapter"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
md_header_splits = markdown_splitter.split_text(docs_as_mark)
md_header_splits

In [ ]:
# Char-level splits
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = 100
chunk_overlap = 20
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="gpt2",
    chunk_size=chunk_size, chunk_overlap=chunk_overlap, separators=['\n-'],
)

# Split
splits = text_splitter.split_documents(md_header_splits)
splits

In [ ]:
from rank_bm25 import BM25Okapi
tokenized_corpus = [doc.page_content.split(" ") for doc in splits]

bm25 = BM25Okapi(tokenized_corpus)

In [32]:
splits

[Document(metadata={'Book': 'Gênesis', 'Chapter': 'Chapter 1'}, page_content='- In the beginning God created the heaven and the earth.\n- And the earth was without form, and void; and darkness was upon the face of the deep. And the Spirit of God moved upon the face of the waters.\n- And God said, Let there be light: and there was light.\n- And God saw the light, that it was good: and God divided the light from the darkness.'),
 Document(metadata={'Book': 'Gênesis', 'Chapter': 'Chapter 1'}, page_content='- And God called the light Day, and the darkness he called Night. And the evening and the morning were the first day.\n- And God said, Let there be a firmament in the midst of the waters, and let it divide the waters from the waters.\n- And God made the firmament, and divided the waters which were under the firmament from the waters which were above the firmament: and it was so.'),
 Document(metadata={'Book': 'Gênesis', 'Chapter': 'Chapter 1'}, page_content='- And God called the firmame

In [ ]:
from sentence_transformers import SentenceTransformer

model_id = "google/embeddinggemma-2"